# GD-CLASS Explorer

**Glassy Dynamics of Spacetime — Interactive Parameter Explorer**

[GitHub repository](https://github.com/lawdroid/class_public/tree/feature/kappa-evolution)

---

### Compliant Inclusion Model (Feb 2026 pivot)

The early vacuum is a **stretched glass** filled with compliant inclusions.
At the Scher-Zallen percolation threshold (phi_c = 0.15):

- **Early universe** (z > z_freeze): kappa = 1 - phi = **0.85** (stronger gravity)
- **Late universe** (z < z_freeze): kappa = **1.0** (standard gravity)
- Transition triggered by **recombination quench** at z ~ 1100

---

### How to use

1. Click **Runtime -> Run all** in the menu bar
2. Wait ~30 seconds for all cells to finish
3. Scroll down to the **Interactive Explorer**
4. **Move the sliders** to change GD parameters
5. Plots and results table update automatically

**No Python knowledge required.**

---

### What this notebook computes

GD theory modifies the Friedmann equation with spacetime stiffness kappa(z):

| | Equation |
|---|---|
| Standard | H^2 = (8piG/3) x rho |
| GD Theory | H^2 = (8piG/3kappa) x rho_matter_rad + Lambda |

The four sliders control:
- **kappa(z) profile** — how stiffness evolves with redshift
- **H(z)** — expansion rate compared to LCDM
- **r_s** — the sound horizon (acoustic ruler)
- **H_0** — implied Hubble constant from r_s ratio


## Setup
Run this cell to load all required libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64, io, warnings
warnings.filterwarnings('ignore')

# Use a clean plot style
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})
print("Setup complete.")

## Load Pre-computed Spectra
Embedded CLASS output and Planck 2018 data (no files needed).

In [ ]:
# Pre-computed CMB power spectra (embedded as compressed data)
_b64 = "..."
_buf = io.BytesIO(base64.b64decode(_b64))
_data = np.load(_buf)

# LCDM baseline (Planck 2018 best-fit, h=0.6736)
l_lcdm = _data['l_lcdm']
Dl_lcdm = _data['Dl_lcdm']

# GD compliant model: kappa_c=0.85, stretched exponential, z_onset=10^6, z_freeze=1100
l_compliant = _data['l_compliant']
Dl_compliant = _data['Dl_compliant']

# GD step model: kappa_c=0.85, sharp step at z=1100
l_step = _data['l_step']
Dl_step = _data['Dl_step']

# Planck 2018 TT observed data
l_planck = _data['l_planck']
Dl_planck = _data['Dl_planck']
err_lo = _data['err_lo']
err_hi = _data['err_hi']

print(f"Loaded: LCDM ({len(l_lcdm)} pts), Compliant ({len(l_compliant)} pts), Step ({len(l_step)} pts), Planck ({len(l_planck)} pts)")


## Physics Engine
The modified Friedmann equation and sound horizon calculation, implemented in Python.
These reproduce the CLASS background computation for any GD parameters.

In [ ]:
# ===== Cosmological parameters (Planck 2018 best-fit) =====
h = 0.6736
H0 = h * 100.0          # km/s/Mpc
c_light = 299792.458     # km/s
omega_b = 0.02237        # Omega_b h^2
omega_cdm = 0.1200       # Omega_cdm h^2

Omega_b = omega_b / h**2
Omega_cdm = omega_cdm / h**2
Omega_m = Omega_b + Omega_cdm

# Radiation: photons + 3 neutrino species (N_eff = 3.044)
Omega_gamma = 2.469e-5 / h**2       # photon density
Omega_r = Omega_gamma * 1.6914      # total radiation (photons + neutrinos)
Omega_Lambda = 1.0 - Omega_m - Omega_r  # flat universe

# ===== kappa(z) compliant inclusion model =====
def kappa_of_z(z, kappa_c, z_freeze, z_onset, beta):
    """GD spacetime stiffness: kappa_c at early times, 1.0 at late times."""
    z = np.atleast_1d(np.float64(z))
    kappa = np.ones_like(z)
    # Above z_onset: frozen at kappa_c (early universe, modified gravity)
    mask_high = z >= z_onset
    kappa[mask_high] = kappa_c
    # Below z_freeze: standard gravity (kappa = 1.0)
    mask_low = z <= z_freeze
    kappa[mask_low] = 1.0
    # Between z_freeze and z_onset: stretched exponential transition
    mask_mid = (z > z_freeze) & (z < z_onset)
    t = (z[mask_mid] - z_freeze) / (z_onset - z_freeze)
    decay = np.exp(-(t / 0.5)**beta)
    kappa[mask_mid] = kappa_c + (1.0 - kappa_c) * decay
    return kappa

# ===== Hubble parameter H(z) =====
def H_of_z(z, kappa_c=1.0, z_freeze=1100, z_onset=1e6, beta=0.5):
    """Hubble parameter in km/s/Mpc with GD modification."""
    z = np.atleast_1d(np.float64(z))
    kap = kappa_of_z(z, kappa_c, z_freeze, z_onset, beta)
    rho_mr = Omega_r * (1+z)**4 + Omega_m * (1+z)**3  # matter + radiation
    rho_L = Omega_Lambda                                # Lambda (constant)
    return H0 * np.sqrt(rho_mr / kap + rho_L)

# ===== Sound horizon r_s =====
def compute_rs(kappa_c, z_freeze, z_onset, beta, z_star=1089.8):
    """Comoving sound horizon at recombination [Mpc]."""
    def integrand(z):
        kap = kappa_of_z(z, kappa_c, z_freeze, z_onset, beta)[0]
        rho_mr = Omega_r * (1+z)**4 + Omega_m * (1+z)**3
        H = H0 * np.sqrt(rho_mr / kap + Omega_Lambda)
        # Baryon loading R = 3*rho_b/(4*rho_gamma)
        R = 3.0 * Omega_b * (1+z)**3 / (4.0 * Omega_gamma * (1+z)**4)
        cs = 1.0 / np.sqrt(3.0 * (1.0 + R))  # sound speed / c
        return cs * c_light / H
    result, _ = quad(integrand, z_star, 1e7, limit=200)
    return result

# ===== Angular diameter distance =====
def compute_DA(kappa_c, z_freeze, z_onset, beta, z_star=1089.8):
    """Comoving angular diameter distance to recombination [Mpc]."""
    def integrand(z):
        return c_light / H_of_z(z, kappa_c, z_freeze, z_onset, beta)[0]
    result, _ = quad(integrand, 0, z_star, limit=200)
    return result

# ===== Implied H_0 from sound horizon ratio =====
def H0_implied(kappa_c, z_freeze, z_onset, beta):
    """Implied H0 if an observer fits LCDM to the modified r_s."""
    rs = compute_rs(kappa_c, z_freeze, z_onset, beta)
    rs_lcdm = compute_rs(1.0, 1100, 1e6, 0.5)
    # H0 ~ 1/r_s at fixed angular scale
    return H0 * rs_lcdm / rs

# ===== Age of universe =====
def compute_age(kappa_c, z_freeze, z_onset, beta):
    """Age of universe [Gyr]."""
    def integrand(z):
        kap = kappa_of_z(z, kappa_c, z_freeze, z_onset, beta)[0]
        rho_mr = Omega_r * (1+z)**4 + Omega_m * (1+z)**3
        H = H0 * np.sqrt(rho_mr / kap + Omega_Lambda)
        return 1.0 / ((1+z) * H)
    # Split integral for better convergence
    result = 0.0
    breaks = [0, 1, 10, 100, 1000, 1e4, 1e5, 1e6, 1e7]
    for i in range(len(breaks)-1):
        val, _ = quad(integrand, breaks[i], breaks[i+1], limit=100)
        result += val
    # Convert: result is Mpc*s/km. 1 Mpc = 3.0857e19 km. 1 Gyr = 3.1557e16 s.
    return result * 3.0857e19 / 3.1557e16

# Test: LCDM values
rs_lcdm = compute_rs(1.0, 1100, 1e6, 0.5)
age_lcdm = compute_age(1.0, 1100, 1e6, 0.5)
print(f"LCDM check:  r_s = {rs_lcdm:.1f} Mpc,  H_0 = {H0:.2f} km/s/Mpc,  Age = {age_lcdm:.2f} Gyr")
print(f"(Expected:   r_s ~ 144.5 Mpc,  H_0 = 67.36 km/s/Mpc,  Age ~ 13.8 Gyr)")


---
## Interactive Explorer

**Move the sliders below** to change GD parameters and see how the physics responds.

| Parameter | Physical meaning |
|-----------|------------------|
| **kappa_c** | Spacetime stiffness in the early universe. 1.0 = standard gravity (LCDM). 0.85 = Scher-Zallen compliant inclusion (stronger gravity). |
| **beta** | Stretched exponent controlling transition shape. Small beta = gradual, large beta = sharp. |
| **z_onset** | Redshift above which kappa = kappa_c. Higher = modification extends deeper into early universe. |
| **z_freeze** | Redshift below which kappa = 1.0 (standard gravity). 1100 = recombination quench. |


In [ ]:
# ===== Create interactive widgets =====
style = {'description_width': '100px'}

slider_kc = widgets.FloatSlider(
    value=0.85, min=0.50, max=1.0, step=0.005,
    description='kappa_c:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.3f'
)
slider_beta = widgets.FloatSlider(
    value=0.5, min=0.1, max=1.0, step=0.05,
    description='beta:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.2f'
)
slider_zonset = widgets.FloatLogSlider(
    value=1e6, min=3, max=8, step=0.1,
    description='z_onset:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.0e'
)
slider_zfreeze = widgets.FloatSlider(
    value=1100, min=100, max=5000, step=50,
    description='z_freeze:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.0f'
)

output = widgets.Output()

def update_plots(change=None):
    kc = slider_kc.value
    beta = slider_beta.value
    zo = slider_zonset.value
    zf = slider_zfreeze.value

    with output:
        clear_output(wait=True)

        # Compute background quantities
        rs = compute_rs(kc, zf, zo, beta)
        rs_ref = compute_rs(1.0, 1100, 1e6, 0.5)  # LCDM reference
        h0_imp = H0 * rs_ref / rs  # H0 implied from r_s ratio
        age = compute_age(kc, zf, zo, beta)
        DA = compute_DA(kc, zf, zo, beta)
        DA_ref = compute_DA(1.0, 1100, 1e6, 0.5)
        theta = rs / DA
        theta_ref = rs_ref / DA_ref

        # ===== FIGURE =====
        fig = plt.figure(figsize=(14, 12))

        # --- Panel 1: kappa(z) profile ---
        ax1 = fig.add_subplot(2, 2, 1)
        z_arr = np.logspace(0, 7, 500)
        kap = kappa_of_z(z_arr, kc, zf, zo, beta)
        ax1.semilogx(z_arr, kap, 'b-', linewidth=2, label=f'GD (kappa_c={kc:.3f})')
        ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='LCDM (kappa=1)')
        ax1.axvline(x=zf, color='red', linestyle=':', alpha=0.5, label=f'z_freeze={zf:.0f}')
        ax1.axvline(x=zo, color='green', linestyle=':', alpha=0.5, label=f'z_onset={zo:.0e}')
        ax1.axvline(x=1100, color='orange', linestyle='--', alpha=0.3, label='Recombination')
        ax1.set_xlabel('Redshift z')
        ax1.set_ylabel('kappa(z)')
        ax1.set_title('Spacetime Stiffness Profile')
        ax1.legend(fontsize=8, loc='lower left')
        ax1.set_ylim(min(kc - 0.05, 0.8), 1.05)

        # --- Panel 2: H(z)/H_LCDM(z) ratio ---
        ax2 = fig.add_subplot(2, 2, 2)
        z_arr2 = np.logspace(0, 5, 300)
        H_gd = H_of_z(z_arr2, kc, zf, zo, beta)
        H_lcdm_arr = H_of_z(z_arr2, 1.0, 1100, 1e6, 0.5)
        ratio = H_gd / H_lcdm_arr
        ax2.semilogx(z_arr2, ratio, 'r-', linewidth=2)
        ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
        ax2.axvline(x=1100, color='orange', linestyle='--', alpha=0.3, label='Recombination')
        ax2.set_xlabel('Redshift z')
        ax2.set_ylabel('H_GD(z) / H_LCDM(z)')
        ax2.set_title('Expansion Rate Ratio')
        ax2.legend(fontsize=8)

        # --- Panel 3: CMB Power Spectrum ---
        ax3 = fig.add_subplot(2, 1, 2)
        ax3.errorbar(l_planck, Dl_planck, yerr=[err_lo, err_hi],
                     fmt='.', color='gray', alpha=0.3, markersize=2,
                     label='Planck 2018', zorder=1)
        ax3.plot(l_lcdm, Dl_lcdm, 'k-', linewidth=1.5, alpha=0.8,
                 label='LCDM (standard)', zorder=2)
        ax3.plot(l_compliant, Dl_compliant, 'b-', linewidth=1.5, alpha=0.8,
                 label='Compliant (kappa=0.85, stretched)', zorder=3)
        ax3.plot(l_step, Dl_step, 'r--', linewidth=1.5, alpha=0.8,
                 label='Step (kappa=0.85, sharp at z=1100)', zorder=3)
        ax3.set_xlabel('Multipole l')
        ax3.set_ylabel('D_l [muK^2]')
        ax3.set_title('CMB TT Power Spectrum (pre-computed CLASS output)')
        ax3.set_xlim(2, 2500)
        ax3.set_ylim(-200, 12000)
        ax3.legend(fontsize=9)

        plt.tight_layout()
        plt.show()

        # ===== RESULTS TABLE =====
        print("=" * 70)
        print("  RESULTS TABLE")
        print("=" * 70)
        print(f"  {'Quantity':<30s} {'LCDM':<15s} {'GD':<15s} {'Change':<10s}")
        print("-" * 70)
        print(f"  {'Sound horizon r_s [Mpc]':<30s} {rs_ref:<15.2f} {rs:<15.2f} {(rs/rs_ref-1)*100:+.2f}%")
        print(f"  {'Ang. diam. dist. D_A [Mpc]':<30s} {DA_ref:<15.1f} {DA:<15.1f} {(DA/DA_ref-1)*100:+.2f}%")
        print(f"  {'theta_* = r_s/D_A':<30s} {theta_ref:<15.6f} {theta:<15.6f} {(theta/theta_ref-1)*100:+.2f}%")
        print(f"  {'Implied H_0 [km/s/Mpc]':<30s} {H0:<15.2f} {h0_imp:<15.2f} {(h0_imp/H0-1)*100:+.2f}%")
        print(f"  {'G_eff/G_N (early, z>z_onset)':<30s} {'1.000':<15s} {1.0/kc:<15.5f} {(1.0/kc-1)*100:+.1f}%")
        print(f"  {'Age [Gyr]':<30s} {age_lcdm:<15.2f} {age:<15.2f} {(age/age_lcdm-1)*100:+.2f}%")
        print("=" * 70)
        print()
        if kc < 0.999:
            print("  NOTE: CMB spectrum plot shows pre-computed CLASS runs.")
            print("  The sliders update background quantities (r_s, H_0, etc.).")
            print("  Full spectrum with your slider values requires running CLASS.")
            print()
            if abs(rs - rs_ref) > 0.1:
                print(f"  KEY INSIGHT: r_s shrinks by {(1-rs/rs_ref)*100:.2f}%")
                print(f"  -> Implied H_0 = {h0_imp:.2f} km/s/Mpc")
                if h0_imp > 70:
                    print(f"  -> In the Hubble tension range! (SH0ES: 73.04 +/- 1.04)")
                else:
                    print(f"  -> Not enough to resolve Hubble tension (need ~73)")
        else:
            print("  kappa_c ~ 1.0: This is standard LCDM (no GD modification).")

# Connect sliders to update function
for s in [slider_kc, slider_beta, slider_zonset, slider_zfreeze]:
    s.observe(update_plots, names='value')

# Display
print("Move the sliders below, then scroll down to see the plots and results.")
print()
display(widgets.VBox([slider_kc, slider_beta, slider_zonset, slider_zfreeze]))
display(output)

# Initial plot
update_plots()


---
## Understanding the Parameters

### kappa_c (Early universe stiffness)
- **kappa_c = 1.0**: Standard gravity. No GD modification. This IS Lambda-CDM.
- **kappa_c = 0.85**: The Scher-Zallen percolation threshold for compliant inclusions
  (phi_c = 0.15, kappa = 1 - phi). This is the value predicted by GD theory.
- **kappa_c < 1**: Stronger effective gravity (G_eff = G_N / kappa > G_N).
  Pre-recombination expansion is faster, shrinking the sound horizon r_s.

### beta (Stretched exponent)
- Controls the **shape** of the kappa(z) transition, NOT its magnitude.
- **Small beta (0.3)**: Most of the change happens near z_onset, with a long gradual tail.
- **Large beta (0.9)**: Change concentrated in the middle of the transition.
- **CRITICAL FINDING**: The transition shape matters enormously! A gradual transition
  (spread over z=1100 to 10^6) barely changes r_s because the sound horizon is
  a late-weighted integral. The modification must be active near z~1100.

### z_onset (Early boundary)
- The redshift above which kappa = kappa_c (modified gravity).
- **z_onset = 10^6**: Conservative. But kappa=0.85 extends to BBN (z~10^9) — needs care.
- **z_onset close to z_freeze**: Sharp step, maximum effect on r_s.

### z_freeze (Late boundary = recombination quench)
- The redshift below which kappa = 1.0 (standard gravity).
- **z_freeze = 1100**: Recombination. Photon decoupling triggers the glass quench.
- This is physically motivated: the acoustic radiation pressure that kept the
  vacuum 'melted' vanishes at photon decoupling.

---

## Key Findings from CLASS Simulations

| Finding | Details |
|---------|--------|
| **Sharp step: H_0 = 73.02** | With kappa=0.85 as a step at z=1100, r_s shrinks 7.75%, giving H_0 = 73.02. Tom's prediction confirmed! |
| **Stretched exponential: H_0 = 67.78** | With gradual transition (z_onset=10^6, beta=0.5), r_s only shrinks 0.62%. The transition shape kills the effect. |
| **ISW under control** | D_l(l=2) ~ 389-1008 muK^2 (compliant model), vs catastrophic 10,441 muK^2 from old rigid model. |
| **Peak structure destroyed (step)** | Sharp step gives right H_0 but peak amplitudes double and positions shift. Phase C (perturbation G_eff) is essential. |
| **Transition shape is key** | r_s is a late-weighted integral. kappa must be active near z~1100, not just at z>>10^5. |
| **Phase C needed** | Background-only kappa cannot fix H_0 AND preserve CMB peaks. Consistent perturbations with G_eff in Poisson equation are the critical next step. |

---
